# PSF Size Across the LSSTCam Focal Plane

Explore `visitSummary` from the `dp2_prep` butler repo to characterise PSF size at the detector level.

**Repo:** `/sdf/group/rubin/repo/dp2_prep`  
**Collection:** `LSSTCam/runs/DRP/DP2`  
**Dataset type:** `visitSummary` — one `ExposureCatalog` per visit, one row per detector

Key PSF column: `psfSigma` (effective Gaussian sigma, pixels).  
Converted to FWHM in arcsec using `psfFWHM = 2.355 × psfSigma × 0.2 arcsec/pix`.

Plots produced:
1. Distribution of per-detector median PSF FWHM across the focal plane (colour map)
2. Per-detector IQR (range) of PSF FWHM across visits

## Setup

In [ ]:
import os
import sys
import importlib

# ── Load the local .env (generated from lsst-scipipe-13.0.0 via create_dot_env.py)
# The local .env next to this notebook matches the kernel's actual stack version.
ENV_FILE = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", ".env")
if not os.path.exists(ENV_FILE):
    # Fallback to ~/notebooks/.env
    ENV_FILE = os.path.join(os.path.expanduser("~"), "notebooks", ".env")

PATH_VARS = {"PYTHONPATH", "PATH", "LD_LIBRARY_PATH"}

if os.path.exists(ENV_FILE):
    env_vals = {}
    with open(ENV_FILE) as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _key, _val = _line.split("=", 1)
            env_vals[_key] = _val
    for _key, _val in env_vals.items():
        if _key in PATH_VARS:
            existing = os.environ.get(_key, "")
            new_entries = [
                p for p in _val.split(":") if p and p not in existing.split(":")
            ]
            if new_entries:
                os.environ[_key] = ":".join(new_entries) + (
                    ":" + existing if existing else ""
                )
        else:
            os.environ.setdefault(_key, _val)
    for _p in reversed(os.environ.get("PYTHONPATH", "").split(":")):
        if _p and _p not in sys.path:
            sys.path.insert(0, _p)
    print(f"Loaded env from {ENV_FILE}")
else:
    print(f"WARNING: {ENV_FILE} not found")

# ── Fix lsst namespace so all EUPS paths are visible ─────────────────────────
for _mod in list(sys.modules.keys()):
    if _mod == "lsst" or _mod.startswith("lsst."):
        del sys.modules[_mod]
importlib.invalidate_caches()
print("Setup complete.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from tqdm.notebook import tqdm

import lsst.daf.butler as daf_butler

print("lsst.daf.butler version:", daf_butler.__version__)
%matplotlib inline

## Butler configuration

In [ ]:
REPO = "/sdf/group/rubin/repo/dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/DP2"

butler = daf_butler.Butler(REPO, collections=[COLLECTION])
print(f"Connected to {REPO}")
print(f"Collection : {COLLECTION}")

## Discover available dataset types (sanity check)

In [ ]:
# Show dataset types that contain 'visit' or 'psf' in the name
registry = butler.registry
all_types = sorted(dt.name for dt in registry.queryDatasetTypes())
relevant = [n for n in all_types if any(kw in n.lower() for kw in ("visit", "psf"))]
print("\n".join(relevant))

## Query visit_detector_table datasets

`visit_detector_table` is a pre-aggregated monolithic table covering all visits × detectors. There are only 2 datasets in this collection (one per instrument run), each with ~5M rows — no per-visit looping needed.

In [ ]:
DATASET_TYPE = "visit_detector_table"

vdt_refs = list(butler.registry.queryDatasets(DATASET_TYPE, collections=[COLLECTION]))
print(f"Found {len(vdt_refs)} {DATASET_TYPE} dataset(s)")
for ref in vdt_refs:
    print(f"  dataId: {ref.dataId}")

## Load visit_detector_table and extract PSF metrics

Each dataset is a pre-aggregated astropy Table (~5M rows). We load all of them and concatenate into a single DataFrame.

In [ ]:
PIXEL_SCALE = 0.2  # arcsec/pixel for LSSTCam

# Load all visit_detector_table datasets (pre-aggregated, no per-visit loop needed)
tables = []
for ref in vdt_refs:
    tbl = butler.get(ref)
    tables.append(tbl.to_pandas())

vsdf = pd.concat(tables, ignore_index=True)
# Source table has both 'detectorId' and 'detector' (same value); also 'visitId' but no 'visit'
vsdf = vsdf.rename(columns={"visitId": "visit"})
vsdf = vsdf.drop(columns=["detectorId"], errors="ignore")

# Compute PSF FWHM in arcsec from psfSigma (pixels)
vsdf["psfFwhm_arcsec"] = 2.355 * vsdf["psfSigma"] * PIXEL_SCALE

# Keep only rows where psfSigma is finite and positive
vsdf = vsdf[vsdf["psfSigma"].gt(0) & vsdf["psfSigma"].notna()].copy()

print(f"Loaded {len(vsdf):,} (visit, detector) rows")
print(
    f"Unique visits: {vsdf['visit'].nunique():,},  unique detectors: {vsdf['detector'].nunique()}"
)
if "band" in vsdf.columns:
    print("Bands:", sorted(vsdf["band"].dropna().unique()))
vsdf[["visit", "detector", "band", "psfSigma", "psfFwhm_arcsec"]].head()

## Per-detector PSF statistics across visits

In [ ]:
det_stats = (
    vsdf.groupby("detector")["psfFwhm_arcsec"]
    .agg(
        n_visits="count",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
        psfFwhm_min="min",
        psfFwhm_max="max",
    )
    .reset_index()
)
print(f"Detectors with data: {len(det_stats)}")
print(det_stats.describe())
det_stats.head(10)

## Fetch focal-plane detector positions from camera geometry

In [ ]:
from lsst.afw.cameraGeom import FOCAL_PLANE

# Load camera geometry from butler (most reliable approach)
camera = butler.get("camera", instrument="LSSTCam")

det_geom = []
for det in camera:
    det_id = det.getId()
    name = det.getName()
    fp_center = det.getCenter(FOCAL_PLANE)
    bb = det.getBBox()
    det_geom.append(
        {
            "detector": det_id,
            "name": name,
            "fp_x": fp_center.getX(),
            "fp_y": fp_center.getY(),
            "width": bb.getWidth(),
            "height": bb.getHeight(),
        }
    )

geom_df = pd.DataFrame(det_geom)
print(f"Camera has {len(geom_df)} detectors")
geom_df.head()

## Focal-plane map: median PSF FWHM per detector

Each rectangle represents one detector, coloured by its median PSF FWHM (arcsec) across all loaded visits.

In [ ]:
# Merge geometry with stats
merged = geom_df.merge(det_stats, on="detector", how="left")

# LSSTCam: 10 μm/pixel → 1 pixel = 0.01 mm
PIX_TO_MM = 0.01

cmap = plt.cm.RdYlGn_r

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, col, title in [
    (axes[0], "median", "Median PSF FWHM (arcsec)"),
    (axes[1], "iqr", "IQR of PSF FWHM across visits (arcsec)"),
]:
    vmin_p = merged[col].quantile(0.02)
    vmax_p = merged[col].quantile(0.98)
    norm_p = mcolors.Normalize(vmin=vmin_p, vmax=vmax_p)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_p)
    sm.set_array([])

    for _, row in merged.iterrows():
        val = row[col]
        color = cmap(norm_p(val)) if pd.notna(val) else "0.7"
        w = row["width"] * PIX_TO_MM
        h = row["height"] * PIX_TO_MM
        rect = Rectangle(
            (row["fp_x"] - w / 2, row["fp_y"] - h / 2),
            w,
            h,
            linewidth=0.3,
            edgecolor="k",
            facecolor=color,
        )
        ax.add_patch(rect)
        ax.text(
            row["fp_x"],
            row["fp_y"],
            f"{val:.2f}" if pd.notna(val) else "",
            ha="center",
            va="center",
            fontsize=4,
            color="k",
        )

    ax.set_aspect("equal")
    ax.autoscale()
    ax.set_xlabel("Focal-plane X (mm)")
    ax.set_ylabel("Focal-plane Y (mm)")
    ax.set_title(title)
    fig.colorbar(sm, ax=ax, label=title, shrink=0.85)

n_visits = vsdf["visit"].nunique()
fig.suptitle(
    f"LSSTCam focal-plane PSF FWHM — {n_visits:,} visits (all bands)\n"
    f"Collection: {COLLECTION}",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## Summary table: detectors sorted by median PSF FWHM

In [ ]:
summary = merged[
    ["detector", "name", "n_visits", "median", "iqr", "psfFwhm_min", "psfFwhm_max"]
].copy()
summary = summary.sort_values("median").reset_index(drop=True)
summary.columns = [
    "detector",
    "name",
    "n_visits",
    "median_fwhm",
    "iqr_fwhm",
    "min_fwhm",
    "max_fwhm",
]
for c in ["median_fwhm", "iqr_fwhm", "min_fwhm", "max_fwhm"]:
    summary[c] = summary[c].round(3)
display(summary)

## Distribution of median PSF FWHM across detectors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(det_stats["median"].dropna(), bins=40, color="steelblue", edgecolor="w")
axes[0].set_xlabel("Median PSF FWHM (arcsec)")
axes[0].set_ylabel("Number of detectors")
axes[0].set_title("Distribution of per-detector median PSF FWHM")

axes[1].hist(det_stats["iqr"].dropna(), bins=40, color="tomato", edgecolor="w")
axes[1].set_xlabel("IQR of PSF FWHM across visits (arcsec)")
axes[1].set_ylabel("Number of detectors")
axes[1].set_title("Distribution of per-detector PSF FWHM IQR")

plt.tight_layout()
plt.show()

print(f"Overall median PSF FWHM: {det_stats['median'].median():.3f} arcsec")
print(
    f"Detector-to-detector spread (std of medians): {det_stats['median'].std():.3f} arcsec"
)